In [ ]:
# ── Paths derived from Snakefile (make_plots_i / make_plots rules) ────────────
import os, sys
from pathlib import Path

# Same env vars as the Snakefile
GWAK_DIR    = Path(os.environ.get('GWAK_DIR',    '/home/katya.govorkova/gwak2/'))
OUTPUT_DIR  = Path(os.environ.get('GWAK_OUTPUT_DIR', GWAK_DIR / 'gwak/output'))

# Wildcards used in the make_plots rule
CL_CONFIG  = 'ResNet'
FM_CONFIG  = 'NF_from_file_conditioning'
IFOS       = 'HL'

# Paths exactly as make_plots_i constructs them
EMBEDDING_MODEL = OUTPUT_DIR / f'{CL_CONFIG}_{IFOS}/model_JIT.pt'
FM_MODEL        = OUTPUT_DIR / f'{CL_CONFIG}_{FM_CONFIG}_{IFOS}/model_JIT.pt'
DATA_DIR        = OUTPUT_DIR / f'BBC_AnalysisReady_Cat12/{IFOS}/'
CONFIG_PATH     = GWAK_DIR   / f'gwak/train/configs/{CL_CONFIG}.yaml'
BURST_BENCHMARK = GWAK_DIR   / 'gwak/scripts/burst_benchmark_short-0.h5'

sys.path.insert(0, str(GWAK_DIR))
sys.path.insert(0, str(GWAK_DIR / 'gwak/train'))

N_PLOT = 6   # number of samples to visualise

for label, p in [('embedding_model', EMBEDDING_MODEL),
                 ('fm_model',         FM_MODEL),
                 ('data_dir',         DATA_DIR),
                 ('config',           CONFIG_PATH),
                 ('burst_benchmark',  BURST_BENCHMARK)]:
    status = '✓' if Path(p).exists() else '✗ NOT FOUND'
    print(f'{status}  {label}: {p}')

## 1. Load model & config (same as plots.py)

In [16]:
import yaml
import numpy as np
import torch
import matplotlib.pyplot as plt

from ml4gw.distributions import PowerLaw
from ml4gw.waveforms import SineGaussian, MultiSineGaussian, IMRPhenomPv2, Gaussian, GenerateString, WhiteNoiseBurst
from gwak.train.dataloader import SignalDataloader
from gwak.data.prior import (
    SineGaussianBBC, MultiSineGaussianBBC, LAL_BBHPrior,
    GaussianBBC, CuspBBC, KinkBBC, KinkkinkBBC, WhiteNoiseBurstBBC
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

embed_model = torch.jit.load(EMBEDDING_MODEL, map_location=device)
embed_model.eval()
print('Model loaded')

with open(CONFIG_PATH, 'r') as f:
    config = yaml.safe_load(f)

cfg              = config['data']['init_args']
sample_rate      = cfg['sample_rate']
kernel_length    = cfg['kernel_length']
psd_length       = cfg['psd_length']
fduration        = cfg['fduration']
fftlength        = cfg['fftlength']
num_workers      = cfg['num_workers']
data_saving_file = cfg.get('data_saving_file', None)
duration         = fduration + kernel_length
batch_size       = 32   # small for notebook

print(f'sample_rate={sample_rate}, kernel_length={kernel_length}s, psd_length={psd_length}s')

device: cpu
Model loaded
sample_rate=4096, kernel_length=1.0s, psd_length=64s


## 2. Set up SignalDataloader (same as plots.py)

In [17]:
signal_classes = [
    'MultiSineGaussian', 'SineGaussian', 'BBH', 'Gaussian',
    'Cusp', 'Kink', 'KinkKink', 'WhiteNoiseBurst',
    'CCSN', 'Background', 'Glitch',
]

priors = [
    MultiSineGaussianBBC(), SineGaussianBBC(), LAL_BBHPrior(), GaussianBBC(),
    CuspBBC(), KinkBBC(), KinkkinkBBC(), WhiteNoiseBurstBBC(),
    None, None, None,
]

waveforms = [
    MultiSineGaussian(sample_rate=sample_rate, duration=duration),
    SineGaussian(sample_rate=sample_rate, duration=duration),
    IMRPhenomPv2(),
    Gaussian(sample_rate=sample_rate, duration=duration),
    GenerateString(sample_rate=sample_rate),
    GenerateString(sample_rate=sample_rate),
    GenerateString(sample_rate=sample_rate),
    WhiteNoiseBurst(sample_rate=sample_rate, duration=duration),
    None, None, None,
]

extra_kwargs = [
    None, None, {'ringdown_duration': 0.9}, None,
    None, None, None, None,
    None, None, None,
]

loader = SignalDataloader(
    signal_classes, priors, waveforms, extra_kwargs,
    data_dir=DATA_DIR,
    sample_rate=sample_rate,
    kernel_length=kernel_length,
    psd_length=psd_length,
    fduration=fduration,
    fftlength=fftlength,
    batch_size=batch_size,
    batches_per_epoch=100,
    num_workers=num_workers,
    data_saving_file=data_saving_file,
    ifos=IFOS,
    snr_prior=PowerLaw(index=3, minimum=4, maximum=30),
)

print('Dataloader ready')

ifos are ['H1', 'L1']
data dir is /home/katya.govorkova/gwak2/gwak/output/BBC_AnalysisReady_Cat12/HL
Dataloader ready


## 3. Get one batch & preprocess (same as plots.py)

In [18]:
test_loader = loader.test_dataloader()
test_iter   = iter(test_loader)

clean_batch, glitch_batch = next(test_iter)
clean_batch  = clean_batch.to(device)
glitch_batch = glitch_batch.to(device)

# on_after_batch_transfer does whitening + bandpass, identical to training
processed, labels, snrs, hrss = loader.on_after_batch_transfer(
    [clean_batch, glitch_batch], None, local_test=True
)

print('processed shape:', processed.shape)   # [batch, num_ifos, kernel_samples]
print('labels shape:',    labels.shape)
print('unique labels:',   labels.unique().cpu().numpy())

/home/katya.govorkova/miniconda3/envs/gwak/lib/python3.11/site-packages/ml4gw/dataloading/hdf5_dataset.py:169: ContiguousHdf5Warning: /home/katya.govorkova/gwak2/gwak/output/BBC_AnalysisReady_Cat12/HL/background-1403650771-4349.h5 stored contiguously – slower I/O
  warnings.warn(f"{fname} stored contiguously – slower I/O", ContiguousHdf5Warning)
/home/katya.govorkova/miniconda3/envs/gwak/lib/python3.11/site-packages/ml4gw/dataloading/hdf5_dataset.py:169: ContiguousHdf5Warning: /home/katya.govorkova/gwak2/gwak/output/BBC_AnalysisReady_Cat12/HL/background-1403655124-6317.h5 stored contiguously – slower I/O
  warnings.warn(f"{fname} stored contiguously – slower I/O", ContiguousHdf5Warning)
/home/katya.govorkova/miniconda3/envs/gwak/lib/python3.11/site-packages/ml4gw/dataloading/hdf5_dataset.py:169: ContiguousHdf5Warning: /home/katya.govorkova/gwak2/gwak/output/BBC_AnalysisReady_Cat12/HL/background-1403661442-4134.h5 stored contiguously – slower I/O
  warnings.warn(f"{fname} stored contigu

processed shape: torch.Size([32, 2, 4096])
labels shape: torch.Size([32])
unique labels: [ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


## 4. Compute embeddings

In [19]:
with torch.no_grad():
    embeddings = embed_model(processed)   # [batch, embed_dim]

processed_np  = processed.detach().cpu().numpy()   # [B, C, T]
embeddings_np = embeddings.detach().cpu().numpy()  # [B, D]
labels_np     = labels.detach().cpu().numpy()
snrs_np       = snrs.detach().cpu().numpy()

print('embeddings shape:', embeddings_np.shape)

embeddings shape: (32, 6)


## 5. Compute embeddings on O4 burst short (from scripts/compute_embeddings.py)

In [ ]:
import h5py
from tqdm import tqdm
from ml4gw.transforms import SpectralDensity, Whiten
from gwak.train.dataloader import TorchBandpassFIR

O4_DIR = OUTPUT_DIR / f'O4_MDC_short-0/{IFOS}/'
o4_files = sorted(O4_DIR.glob('*.h5'))
print(f'Found {len(o4_files)} O4 files in {O4_DIR}')

# Preprocessing transforms (same as compute_embeddings.py)
bandpass = TorchBandpassFIR(lowcut=30, highcut=2047, sample_rate=sample_rate).to(device)
whitener = Whiten(fduration, sample_rate, highpass=30).to(device)
spectral_density = SpectralDensity(sample_rate, fftlength, average='median').to(device)

kernel_samples    = int(kernel_length * sample_rate)
psd_samples       = int(psd_length * sample_rate)
fduration_samples = int(fduration * sample_rate)
batch_size_emb    = 32

def embed_file(h5_path, model, bandpass, whitener, spectral_density, device):
    """Segment, whiten and embed one h5 strain file.

    Returns
    -------
    embeddings   : np.ndarray  [N, D]
    segment_gps  : np.ndarray  [N]   GPS start time of each 1-s window
    """
    with h5py.File(h5_path, 'r') as f:
        ifos_in_file = [k for k in ['H1', 'L1', 'V1'] if k in f]
        strain = np.stack([f[ifo][:] for ifo in ifos_in_file], axis=0)  # [C, T]
        # GPS start time stored as attribute on the first IFO dataset
        gps_start = float(f[ifos_in_file[0]].attrs.get('start_time', 0.0))

    total = strain.shape[1]
    first_start = psd_samples
    available   = total - psd_samples - fduration_samples
    n_segments  = available // kernel_samples

    segments, psd_segs, seg_gps = [], [], []
    for i in range(n_segments):
        start = first_start + i * kernel_samples
        end   = start + kernel_samples + fduration_samples
        if end > total:
            break
        segments.append(strain[:, start:end])
        psd_segs.append(strain[:, start - psd_samples:start])
        seg_gps.append(gps_start + start / sample_rate)

    if not segments:
        return np.array([]), np.array([])

    all_emb = []
    with torch.no_grad():
        for b in range(0, len(segments), batch_size_emb):
            seg_t = torch.tensor(np.stack(segments[b:b+batch_size_emb]), dtype=torch.float32, device=device)
            psd_t = torch.tensor(np.stack(psd_segs[b:b+batch_size_emb]), dtype=torch.float32, device=device)

            seg_bp   = bandpass(seg_t)
            psds     = spectral_density(psd_t.double())
            whitened = whitener(seg_bp.double(), psds.double()).float()
            whitened = whitened[:, :, :kernel_samples]

            stds = whitened.std(dim=-1, keepdim=True).clamp(min=1e-8)
            whitened = whitened / stds

            emb = model(whitened)
            all_emb.append(emb.cpu().numpy())

    return np.concatenate(all_emb, axis=0), np.array(seg_gps)

o4_embeddings_list, o4_gps_list = [], []
for f in tqdm(o4_files, desc='O4 files'):
    emb, gps = embed_file(f, embed_model, bandpass, whitener, spectral_density, device)
    if len(emb):
        o4_embeddings_list.append(emb)
        o4_gps_list.append(gps)

o4_emb      = np.concatenate(o4_embeddings_list, axis=0)
o4_gps      = np.concatenate(o4_gps_list, axis=0)
print(f'O4 embeddings shape: {o4_emb.shape}  GPS range: {o4_gps[0]:.1f} – {o4_gps[-1]:.1f}')

## 6. Background embeddings (from SignalDataloader batch)

## 5b. Read injection metadata from burst benchmark & match to segment indices

In [ ]:
# ── Read injections from burst benchmark file (same pattern as user snippet) ──
inj_type  = []   # str per injection
inj_gps   = []   # GPS time per injection
inj_hrss  = []   # hrss per injection

with h5py.File(BURST_BENCHMARK, 'r') as h5_file:
    for key in list(h5_file.keys()):
        sig_type = h5_file[key].attrs['type']
        params   = h5_file[key]['PARAMETERS'][:]
        times    = params['time'][:]
        hrss_vals = params['hrss'][:]
        for t, h in zip(times, hrss_vals):
            inj_type.append(sig_type)
            inj_gps.append(float(t))
            inj_hrss.append(float(h))

inj_type  = np.array(inj_type)
inj_gps   = np.array(inj_gps)
inj_hrss  = np.array(inj_hrss)
print(f'Total injections: {len(inj_gps)}')
print('Signal types:', np.unique(inj_type))

# ── Match each injection GPS time to the nearest segment index ──────────────
# A 1-s window at index i covers GPS [o4_gps[i], o4_gps[i] + kernel_length)
# We assign the injection to the window whose start is closest.
inj_sample_idx = np.full(len(inj_gps), -1, dtype=int)   # -1 = outside coverage

for j, t in enumerate(inj_gps):
    # find window where t falls: o4_gps[i] <= t < o4_gps[i] + kernel_length
    candidates = np.where((o4_gps <= t) & (t < o4_gps + kernel_length))[0]
    if len(candidates):
        inj_sample_idx[j] = candidates[0]

matched = inj_sample_idx >= 0
print(f'Injections matched to a segment: {matched.sum()} / {len(inj_gps)}')

# ── Build per-segment injection arrays (NaN / empty string if no injection) ─
n_seg           = len(o4_emb)
seg_inj_type    = np.full(n_seg, '',    dtype=object)
seg_inj_hrss    = np.full(n_seg, np.nan)
seg_has_inj     = np.zeros(n_seg, dtype=bool)

for j in np.where(matched)[0]:
    idx = inj_sample_idx[j]
    seg_inj_type[idx] = inj_type[j]
    seg_inj_hrss[idx] = inj_hrss[j]
    seg_has_inj[idx]  = True

print(f'Segments with injection: {seg_has_inj.sum()}')

In [ ]:
# ── Save O4 embeddings + injection metadata to HDF5 ─────────────────────────
out_path = OUTPUT_DIR / 'embeddings/o4_short-0_embeddings_with_injections.h5'
out_path.parent.mkdir(parents=True, exist_ok=True)

with h5py.File(out_path, 'w') as f:
    f.create_dataset('embeddings',   data=o4_emb,                  compression='gzip')
    f.create_dataset('gps_times',    data=o4_gps,                  compression='gzip')
    f.create_dataset('sample_index', data=np.arange(n_seg),        compression='gzip')
    f.create_dataset('has_injection',data=seg_has_inj.astype(np.int8), compression='gzip')
    f.create_dataset('inj_hrss',     data=seg_inj_hrss,            compression='gzip')
    # store type strings as fixed-length bytes
    dt = h5py.special_dtype(vlen=str)
    ds = f.create_dataset('inj_type', shape=(n_seg,), dtype=dt)
    ds[:] = seg_inj_type.tolist()

print(f'Saved to {out_path}')
print(f'  embeddings:    {o4_emb.shape}')
print(f'  injected segs: {seg_has_inj.sum()}  ({seg_has_inj.mean()*100:.1f}%)')

In [ ]:
# Collect background-only embeddings from multiple batches
all_background = ['Background', 'Glitch']
bg_labels_set  = set(i + 1 for i, c in enumerate(signal_classes) if c in all_background)

bkg_emb_list = []
n_batches_bkg = 20   # collect ~20 * batch_size background windows

test_loader = loader.test_dataloader()
test_iter   = iter(test_loader)

for _ in tqdm(range(n_batches_bkg), desc='background batches'):
    try:
        clean_batch, glitch_batch = next(test_iter)
    except StopIteration:
        break
    clean_batch  = clean_batch.to(device)
    glitch_batch = glitch_batch.to(device)

    processed, labels, snrs, hrss = loader.on_after_batch_transfer(
        [clean_batch, glitch_batch], None, local_test=True
    )

    mask = torch.tensor([l.item() in bg_labels_set for l in labels], device=device)
    if mask.sum() == 0:
        continue

    with torch.no_grad():
        emb = embed_model(processed[mask])
    bkg_emb_list.append(emb.cpu().numpy())

bkg_emb = np.concatenate(bkg_emb_list, axis=0)
print(f'Background embeddings shape: {bkg_emb.shape}')

## 7. Per-dimension histogram: background vs O4 burst short

In [ ]:
n_dims = bkg_emb.shape[1]
ncols  = 3
nrows  = (n_dims + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for d in range(n_dims):
    ax = axes[d]
    lo = min(bkg_emb[:, d].min(), o4_emb[:, d].min())
    hi = max(bkg_emb[:, d].max(), o4_emb[:, d].max())
    bins = np.linspace(lo, hi, 60)

    ax.hist(bkg_emb[:, d], bins=bins, density=True, alpha=0.6, label='Background')
    ax.hist(o4_emb[:, d],  bins=bins, density=True, alpha=0.6, label='O4 burst short')
    ax.set_title(f'dim {d}')
    ax.set_xlabel('embedding value')
    ax.set_ylabel('density')
    ax.legend(fontsize=8)

# Hide unused axes
for d in range(n_dims, len(axes)):
    axes[d].set_visible(False)

plt.suptitle('Per-dimension embedding distributions: Background vs O4 burst short', y=1.02)
plt.tight_layout()
plt.show()